In [1]:
import numpy as np

data = np.genfromtxt(
    "../data/ETP_4_New_Data_Accidents.csv",
    delimiter=",",
    dtype=None,
    encoding="utf-8",
    names=True
)

# Exact deduplication
clean_data = np.unique(data)

print("Clean records:", len(clean_data))

Clean records: 4053


In [2]:
chainage = clean_data["Accident_Location_A_Chainage_km"]
severity = clean_data["Accident_Severity_C"]

segments = np.floor(chainage / 10).astype(int)

segment_values, segment_counts = np.unique(
    segments,
    return_counts=True
)

print("Occupied segments:", len(segment_values))
print("Total accidents:", len(clean_data))

Occupied segments: 57
Total accidents: 4053


In [3]:
mean_accidents = segment_counts.mean()
median_accidents = np.median(segment_counts)
std_accidents = segment_counts.std()

print("===== SEGMENT BASELINE =====")

print("Mean accidents per segment:", mean_accidents)
print("Median accidents per segment:", median_accidents)
print("Standard deviation:", std_accidents)

===== SEGMENT BASELINE =====
Mean accidents per segment: 71.10526315789474
Median accidents per segment: 36.0
Standard deviation: 84.36671918258114


In [4]:
segment_z_scores = (
    segment_counts - mean_accidents
) / std_accidents

print("Minimum z-score:", segment_z_scores.min())
print("Maximum z-score:", segment_z_scores.max())

top_z_index = np.argmax(segment_z_scores)

print(
    "\nHighest concentration:",
    segment_values[top_z_index] * 10,
    "-",
    segment_values[top_z_index] * 10 + 10,
    "km"
)

print(
    "Accidents:",
    segment_counts[top_z_index]
)

print(
    "Z-score:",
    segment_z_scores[top_z_index]
)

Minimum z-score: -0.8309587457843104
Maximum z-score: 4.69254631065284

Highest concentration: 490 - 500 km
Accidents: 467
Z-score: 4.69254631065284


In [5]:
segment_mean_severity = np.array([
    severity[segments == segment].mean()
    for segment in segment_values
])

print("Minimum mean severity:", segment_mean_severity.min())
print("Maximum mean severity:", segment_mean_severity.max())

highest_severity_index = np.argmax(segment_mean_severity)

print(
    "\nHighest mean-severity segment:",
    segment_values[highest_severity_index] * 10,
    "-",
    segment_values[highest_severity_index] * 10 + 10,
    "km"
)

print(
    "Mean severity:",
    segment_mean_severity[highest_severity_index]
)

Minimum mean severity: 1.5
Maximum mean severity: 3.2

Highest mean-severity segment: 100 - 110 km
Mean severity: 3.2


In [6]:
overall_mean_severity = severity.mean()

print("Overall mean severity:", overall_mean_severity)

Overall mean severity: 2.6434739698988405


In [7]:
min_z = segment_z_scores.min()
max_z = segment_z_scores.max()

concentration_score = (
    (segment_z_scores - min_z)
    / (max_z - min_z)
)

print(
    "Concentration score:",
    concentration_score.min(),
    "→",
    concentration_score.max()
)

Concentration score: 0.0 → 1.0


In [8]:
min_severity = segment_mean_severity.min()
max_severity = segment_mean_severity.max()

severity_score = (
    (segment_mean_severity - min_severity)
    / (max_severity - min_severity)
)

print(
    "Severity score:",
    severity_score.min(),
    "→",
    severity_score.max()
)

Severity score: 0.0 → 1.0


In [9]:
risk_indicator = (
    0.5 * concentration_score
    + 0.5 * severity_score
)

print("Minimum indicator:", risk_indicator.min())
print("Maximum indicator:", risk_indicator.max())

Minimum indicator: 0.0010729613733905569
Maximum indicator: 0.7679808540118402


In [10]:
ranked_indices = np.argsort(risk_indicator)[::-1]

print("===== TOP 10 MARGGANIT SEGMENTS =====")

for index in ranked_indices[:10]:

    segment = segment_values[index]

    print(
        f"{segment * 10:3d}-{segment * 10 + 10:3d} km | "
        f"Indicator: {risk_indicator[index]:.3f} | "
        f"Concentration: {concentration_score[index]:.3f} | "
        f"Severity: {severity_score[index]:.3f}"
    )

===== TOP 10 MARGGANIT SEGMENTS =====
490-500 km | Indicator: 0.768 | Concentration: 1.000 | Severity: 0.536
240-250 km | Indicator: 0.587 | Concentration: 0.311 | Severity: 0.862
220-230 km | Indicator: 0.577 | Concentration: 0.431 | Severity: 0.722
210-220 km | Indicator: 0.570 | Concentration: 0.367 | Severity: 0.773
500-510 km | Indicator: 0.564 | Concentration: 0.500 | Severity: 0.628
180-190 km | Indicator: 0.563 | Concentration: 0.421 | Severity: 0.706
190-200 km | Indicator: 0.550 | Concentration: 0.258 | Severity: 0.843
200-210 km | Indicator: 0.547 | Concentration: 0.260 | Severity: 0.834
230-240 km | Indicator: 0.541 | Concentration: 0.369 | Severity: 0.712
170-180 km | Indicator: 0.539 | Concentration: 0.378 | Severity: 0.700


In [13]:
weights = [0.3, 0.5, 0.7]

for concentration_weight in weights:

    severity_weight = 1 - concentration_weight

    test_score = (
        concentration_weight * concentration_score
        + severity_weight * severity_score
    )

    top_index = np.argmax(test_score)

    print(
        f"\nConcentration weight: {concentration_weight:.1f}"
    )
    print(
        f"Severity weight:      {severity_weight:.1f}"
    )
    print(
        f"Top segment: "
        f"{segment_values[top_index] * 10}-"
        f"{segment_values[top_index] * 10 + 10} km"
    )
    print(
        f"Score: {test_score[top_index]:.3f}"
    )


Concentration weight: 0.3
Severity weight:      0.7
Top segment: 100-110 km
Score: 0.715

Concentration weight: 0.5
Severity weight:      0.5
Top segment: 490-500 km
Score: 0.768

Concentration weight: 0.7
Severity weight:      0.3
Top segment: 490-500 km
Score: 0.861


In [14]:
times = clean_data["Time_of_Accident"]

hours = np.array([
    int(time.split(":")[0])
    for time in times
])

hour_values, hour_counts = np.unique(
    hours,
    return_counts=True
)

top_hour_indices = np.argsort(hour_counts)[-3:][::-1]

peak_hours = hour_values[top_hour_indices]

print("Top 3 accident hours:", peak_hours)

Top 3 accident hours: [19 16 20]


In [15]:
temporal_concentration = np.array([
    np.mean(
        np.isin(
            hours[segments == segment],
            peak_hours
        )
    )
    for segment in segment_values
])

print(
    "Minimum temporal concentration:",
    temporal_concentration.min()
)

print(
    "Maximum temporal concentration:",
    temporal_concentration.max()
)

Minimum temporal concentration: 0.0
Maximum temporal concentration: 1.0


In [16]:
min_temporal = temporal_concentration.min()
max_temporal = temporal_concentration.max()

temporal_score = (
    (temporal_concentration - min_temporal)
    / (max_temporal - min_temporal)
)

print(
    "Temporal score:",
    temporal_score.min(),
    "→",
    temporal_score.max()
)

Temporal score: 0.0 → 1.0


In [17]:
risk_indicator_v2 = (
    concentration_score
    + severity_score
    + temporal_score
) / 3

print("Minimum indicator:", risk_indicator_v2.min())
print("Maximum indicator:", risk_indicator_v2.max())

Minimum indicator: 0.0007153075822603712
Maximum indicator: 0.5590964437166729


In [18]:
ranked_indices = np.argsort(risk_indicator_v2)[::-1]

print("===== TOP 10 MARGGANIT SEGMENTS =====")

for index in ranked_indices[:10]:

    segment = segment_values[index]

    print(
        f"{segment * 10:3d}-{segment * 10 + 10:3d} km | "
        f"Indicator: {risk_indicator_v2[index]:.3f} | "
        f"Concentration: {concentration_score[index]:.3f} | "
        f"Severity: {severity_score[index]:.3f} | "
        f"Temporal: {temporal_score[index]:.3f}"
    )

===== TOP 10 MARGGANIT SEGMENTS =====
490-500 km | Indicator: 0.559 | Concentration: 1.000 | Severity: 0.536 | Temporal: 0.141
240-250 km | Indicator: 0.446 | Concentration: 0.311 | Severity: 0.862 | Temporal: 0.164
180-190 km | Indicator: 0.440 | Concentration: 0.421 | Severity: 0.706 | Temporal: 0.193
220-230 km | Indicator: 0.439 | Concentration: 0.431 | Severity: 0.722 | Temporal: 0.163
340-350 km | Indicator: 0.431 | Concentration: 0.000 | Severity: 0.294 | Temporal: 1.000
230-240 km | Indicator: 0.426 | Concentration: 0.369 | Severity: 0.712 | Temporal: 0.197
190-200 km | Indicator: 0.425 | Concentration: 0.258 | Severity: 0.843 | Temporal: 0.174
500-510 km | Indicator: 0.419 | Concentration: 0.500 | Severity: 0.628 | Temporal: 0.128
210-220 km | Indicator: 0.419 | Concentration: 0.367 | Severity: 0.773 | Temporal: 0.116
110-120 km | Indicator: 0.416 | Concentration: 0.064 | Severity: 0.958 | Temporal: 0.226


In [19]:
hour_distribution = np.zeros(
    (len(segment_values), 24)
)

for i, segment in enumerate(segment_values):

    segment_hours = hours[segments == segment]

    values, counts = np.unique(
        segment_hours,
        return_counts=True
    )

    hour_distribution[i, values] = counts

print("Shape:", hour_distribution.shape)

Shape: (57, 24)


In [20]:
hour_probabilities = (
    hour_distribution
    / hour_distribution.sum(axis=1, keepdims=True)
)

print(
    "Probability range:",
    hour_probabilities.min(),
    "→",
    hour_probabilities.max()
)

Probability range: 0.0 → 1.0


In [21]:
# Avoid log(0) because 0 * log(0) is undefined.
safe_probabilities = np.where(
    hour_probabilities > 0,
    hour_probabilities,
    1
)

hour_entropy = -np.sum(
    hour_probabilities * np.log2(safe_probabilities),
    axis=1
)

print("Minimum entropy:", hour_entropy.min())
print("Maximum entropy:", hour_entropy.max())

Minimum entropy: -0.0
Maximum entropy: 4.50257999029292


In [22]:
min_entropy = hour_entropy.min()
max_entropy = hour_entropy.max()

temporal_concentration_score = (
    1
    - (
        (hour_entropy - min_entropy)
        / (max_entropy - min_entropy)
    )
)

print(
    "Temporal concentration:",
    temporal_concentration_score.min(),
    "→",
    temporal_concentration_score.max()
)

Temporal concentration: 0.0 → 1.0


In [23]:
temporal_ranking = np.argsort(
    temporal_concentration_score
)[::-1]

print("Top 10 temporally concentrated segments:\n")

for rank, index in enumerate(temporal_ranking[:10], start=1):
    start_km = segment_values[index] * 10
    end_km = start_km + 10

    print(
        f"{rank}. {start_km:.0f}-{end_km:.0f} km"
        f" | Temporal Score: {temporal_concentration_score[index]:.3f}"
        f" | Entropy: {hour_entropy[index]:.3f}"
    )

Top 10 temporally concentrated segments:

1. 720-730 km | Temporal Score: 1.000 | Entropy: -0.000
2. 570-580 km | Temporal Score: 1.000 | Entropy: -0.000
3. 340-350 km | Temporal Score: 1.000 | Entropy: -0.000
4. 740-750 km | Temporal Score: 0.796 | Entropy: 0.918
5. 390-400 km | Temporal Score: 0.784 | Entropy: 0.971
6. 380-390 km | Temporal Score: 0.778 | Entropy: 1.000
7. 650-660 km | Temporal Score: 0.778 | Entropy: 1.000
8. 730-740 km | Temporal Score: 0.648 | Entropy: 1.585
9. 320-330 km | Temporal Score: 0.602 | Entropy: 1.792
10. 80-90 km | Temporal Score: 0.602 | Entropy: 1.792


In [24]:
temporal_ranking = np.argsort(
    temporal_concentration_score
)[::-1]

print("Top 10 temporally concentrated segments:\n")

for rank, index in enumerate(temporal_ranking[:10], start=1):
    start_km = segment_values[index] * 10
    end_km = start_km + 10

    accident_count = segment_counts[index]

    print(
        f"{rank}. {start_km:.0f}-{end_km:.0f} km"
        f" | Accidents: {accident_count}"
        f" | Temporal Score: {temporal_concentration_score[index]:.3f}"
        f" | Entropy: {hour_entropy[index]:.3f}"
    )

Top 10 temporally concentrated segments:

1. 720-730 km | Accidents: 1 | Temporal Score: 1.000 | Entropy: -0.000
2. 570-580 km | Accidents: 1 | Temporal Score: 1.000 | Entropy: -0.000
3. 340-350 km | Accidents: 1 | Temporal Score: 1.000 | Entropy: -0.000
4. 740-750 km | Accidents: 3 | Temporal Score: 0.796 | Entropy: 0.918
5. 390-400 km | Accidents: 5 | Temporal Score: 0.784 | Entropy: 0.971
6. 380-390 km | Accidents: 2 | Temporal Score: 0.778 | Entropy: 1.000
7. 650-660 km | Accidents: 2 | Temporal Score: 0.778 | Entropy: 1.000
8. 730-740 km | Accidents: 3 | Temporal Score: 0.648 | Entropy: 1.585
9. 320-330 km | Accidents: 6 | Temporal Score: 0.602 | Entropy: 1.792
10. 80-90 km | Accidents: 6 | Temporal Score: 0.602 | Entropy: 1.792


In [25]:
k = 10

temporal_reliability = (
    segment_counts / (segment_counts + k)
)

adjusted_temporal_score = (
    temporal_concentration_score
    * temporal_reliability
)

print(
    "Adjusted temporal score:",
    adjusted_temporal_score.min(),
    "→",
    adjusted_temporal_score.max()
)

Adjusted temporal score: 0.0 → 0.2614522787270162


In [26]:
adjusted_ranking = np.argsort(
    adjusted_temporal_score
)[::-1]

print("Top 10 adjusted temporal segments:\n")

for rank, index in enumerate(adjusted_ranking[:10], start=1):
    start_km = segment_values[index] * 10
    end_km = start_km + 10

    print(
        f"{rank}. {start_km:.0f}-{end_km:.0f} km"
        f" | Accidents: {segment_counts[index]}"
        f" | Raw Temporal: {temporal_concentration_score[index]:.3f}"
        f" | Adjusted: {adjusted_temporal_score[index]:.3f}"
    )

Top 10 adjusted temporal segments:

1. 390-400 km | Accidents: 5 | Raw Temporal: 0.784 | Adjusted: 0.261
2. 80-90 km | Accidents: 6 | Raw Temporal: 0.602 | Adjusted: 0.226
3. 320-330 km | Accidents: 6 | Raw Temporal: 0.602 | Adjusted: 0.226
4. 0-10 km | Accidents: 19 | Raw Temporal: 0.308 | Adjusted: 0.202
5. 60-70 km | Accidents: 15 | Raw Temporal: 0.321 | Adjusted: 0.193
6. 710-720 km | Accidents: 5 | Raw Temporal: 0.573 | Adjusted: 0.191
7. 260-270 km | Accidents: 10 | Raw Temporal: 0.368 | Adjusted: 0.184
8. 740-750 km | Accidents: 3 | Raw Temporal: 0.796 | Adjusted: 0.184
9. 660-670 km | Accidents: 7 | Raw Temporal: 0.440 | Adjusted: 0.181
10. 90-100 km | Accidents: 7 | Raw Temporal: 0.440 | Adjusted: 0.181


In [27]:
risk_indicator_final = (
    concentration_score
    + severity_score
    + adjusted_temporal_score
) / 3

print(
    "Final risk indicator:",
    risk_indicator_final.min(),
    "→",
    risk_indicator_final.max()
)

Final risk indicator: 0.043932258232333425 → 0.5125410395773377


In [28]:
final_ranking = np.argsort(
    risk_indicator_final
)[::-1]

print("Top 15 MargGanit risk segments:\n")

for rank, index in enumerate(final_ranking[:15], start=1):
    start_km = segment_values[index] * 10
    end_km = start_km + 10

    print(
        f"{rank}. {start_km:.0f}-{end_km:.0f} km"
        f" | Score: {risk_indicator_final[index]:.3f}"
        f" | Accidents: {segment_counts[index]}"
        f" | Concentration: {concentration_score[index]:.3f}"
        f" | Severity: {severity_score[index]:.3f}"
        f" | Temporal: {adjusted_temporal_score[index]:.3f}"
    )

Top 15 MargGanit risk segments:

1. 490-500 km | Score: 0.513 | Accidents: 467 | Concentration: 1.000 | Severity: 0.536 | Temporal: 0.002
2. 260-270 km | Score: 0.401 | Accidents: 10 | Concentration: 0.019 | Severity: 1.000 | Temporal: 0.184
3. 240-250 km | Score: 0.391 | Accidents: 146 | Concentration: 0.311 | Severity: 0.862 | Temporal: 0.000
4. 280-290 km | Score: 0.390 | Accidents: 5 | Concentration: 0.009 | Severity: 1.000 | Temporal: 0.161
5. 100-110 km | Score: 0.389 | Accidents: 25 | Concentration: 0.052 | Severity: 1.000 | Temporal: 0.114
6. 220-230 km | Score: 0.385 | Accidents: 202 | Concentration: 0.431 | Severity: 0.722 | Temporal: 0.001
7. 390-400 km | Score: 0.384 | Accidents: 5 | Concentration: 0.009 | Severity: 0.882 | Temporal: 0.261
8. 210-220 km | Score: 0.380 | Accidents: 172 | Concentration: 0.367 | Severity: 0.773 | Temporal: 0.001
9. 180-190 km | Score: 0.380 | Accidents: 197 | Concentration: 0.421 | Severity: 0.706 | Temporal: 0.014
10. 500-510 km | Score: 0.37

In [29]:
severity_ranking = np.argsort(
    severity_score
)[::-1]

print("Top 15 severity segments:\n")

for rank, index in enumerate(severity_ranking[:15], start=1):
    start_km = segment_values[index] * 10
    end_km = start_km + 10

    print(
        f"{rank}. {start_km:.0f}-{end_km:.0f} km"
        f" | Accidents: {segment_counts[index]}"
        f" | Mean Severity: {segment_mean_severity[index]:.3f}"
        f" | Severity Score: {severity_score[index]:.3f}"
    )

Top 15 severity segments:

1. 280-290 km | Accidents: 5 | Mean Severity: 3.200 | Severity Score: 1.000
2. 260-270 km | Accidents: 10 | Mean Severity: 3.200 | Severity Score: 1.000
3. 100-110 km | Accidents: 25 | Mean Severity: 3.200 | Severity Score: 1.000
4. 110-120 km | Accidents: 31 | Mean Severity: 3.129 | Severity Score: 0.958
5. 70-80 km | Accidents: 16 | Mean Severity: 3.062 | Severity Score: 0.919
6. 390-400 km | Accidents: 5 | Mean Severity: 3.000 | Severity Score: 0.882
7. 570-580 km | Accidents: 1 | Mean Severity: 3.000 | Severity Score: 0.882
8. 90-100 km | Accidents: 7 | Mean Severity: 3.000 | Severity Score: 0.882
9. 240-250 km | Accidents: 146 | Mean Severity: 2.966 | Severity Score: 0.862
10. 190-200 km | Accidents: 121 | Mean Severity: 2.934 | Severity Score: 0.843
11. 60-70 km | Accidents: 15 | Mean Severity: 2.933 | Severity Score: 0.843
12. 200-210 km | Accidents: 122 | Mean Severity: 2.918 | Severity Score: 0.834
13. 140-150 km | Accidents: 28 | Mean Severity: 2.89

In [30]:
severity_reliability = (
    segment_counts / (segment_counts + k)
)

adjusted_severity_score = (
    severity_score
    * severity_reliability
)

print(
    "Adjusted severity score:",
    adjusted_severity_score.min(),
    "→",
    adjusted_severity_score.max()
)

Adjusted severity score: 0.0 → 0.8069381598793364


In [31]:
adjusted_severity_ranking = np.argsort(
    adjusted_severity_score
)[::-1]

print("Top 15 adjusted severity segments:\n")

for rank, index in enumerate(
    adjusted_severity_ranking[:15],
    start=1
):
    start_km = segment_values[index] * 10
    end_km = start_km + 10

    print(
        f"{rank}. {start_km:.0f}-{end_km:.0f} km"
        f" | Accidents: {segment_counts[index]}"
        f" | Mean Severity: {segment_mean_severity[index]:.3f}"
        f" | Raw: {severity_score[index]:.3f}"
        f" | Adjusted: {adjusted_severity_score[index]:.3f}"
    )

Top 15 adjusted severity segments:

1. 240-250 km | Accidents: 146 | Mean Severity: 2.966 | Raw: 0.862 | Adjusted: 0.807
2. 190-200 km | Accidents: 121 | Mean Severity: 2.934 | Raw: 0.843 | Adjusted: 0.779
3. 200-210 km | Accidents: 122 | Mean Severity: 2.918 | Raw: 0.834 | Adjusted: 0.771
4. 210-220 km | Accidents: 172 | Mean Severity: 2.814 | Raw: 0.773 | Adjusted: 0.730
5. 110-120 km | Accidents: 31 | Mean Severity: 3.129 | Raw: 0.958 | Adjusted: 0.725
6. 100-110 km | Accidents: 25 | Mean Severity: 3.200 | Raw: 1.000 | Adjusted: 0.714
7. 220-230 km | Accidents: 202 | Mean Severity: 2.728 | Raw: 0.722 | Adjusted: 0.688
8. 160-170 km | Accidents: 117 | Mean Severity: 2.761 | Raw: 0.742 | Adjusted: 0.683
9. 230-240 km | Accidents: 173 | Mean Severity: 2.711 | Raw: 0.712 | Adjusted: 0.673
10. 180-190 km | Accidents: 197 | Mean Severity: 2.701 | Raw: 0.706 | Adjusted: 0.672
11. 170-180 km | Accidents: 177 | Mean Severity: 2.689 | Raw: 0.700 | Adjusted: 0.662
12. 410-420 km | Accidents: 1

In [32]:
risk_indicator_final = (
    concentration_score
    + adjusted_severity_score
    + adjusted_temporal_score
) / 3

print(
    "Final risk indicator:",
    risk_indicator_final.min(),
    "→",
    risk_indicator_final.max()
)

Final risk indicator: 0.0392156862745098 → 0.508795674741393


In [33]:
final_ranking = np.argsort(
    risk_indicator_final
)[::-1]

print("Top 15 MargGanit risk segments:\n")

for rank, index in enumerate(
    final_ranking[:15],
    start=1
):
    start_km = segment_values[index] * 10
    end_km = start_km + 10

    print(
        f"{rank}. {start_km:.0f}-{end_km:.0f} km"
        f" | Score: {risk_indicator_final[index]:.3f}"
        f" | Accidents: {segment_counts[index]}"
        f" | Spatial: {concentration_score[index]:.3f}"
        f" | Severity: {adjusted_severity_score[index]:.3f}"
        f" | Temporal: {adjusted_temporal_score[index]:.3f}"
    )

Top 15 MargGanit risk segments:

1. 490-500 km | Score: 0.509 | Accidents: 467 | Spatial: 1.000 | Severity: 0.525 | Temporal: 0.002
2. 220-230 km | Score: 0.374 | Accidents: 202 | Spatial: 0.431 | Severity: 0.688 | Temporal: 0.001
3. 240-250 km | Score: 0.373 | Accidents: 146 | Spatial: 0.311 | Severity: 0.807 | Temporal: 0.000
4. 500-510 km | Score: 0.370 | Accidents: 234 | Spatial: 0.500 | Severity: 0.603 | Temporal: 0.008
5. 180-190 km | Score: 0.369 | Accidents: 197 | Spatial: 0.421 | Severity: 0.672 | Temporal: 0.014
6. 210-220 km | Score: 0.366 | Accidents: 172 | Spatial: 0.367 | Severity: 0.730 | Temporal: 0.001
7. 190-200 km | Score: 0.357 | Accidents: 121 | Spatial: 0.258 | Severity: 0.779 | Temporal: 0.035
8. 170-180 km | Score: 0.352 | Accidents: 177 | Spatial: 0.378 | Severity: 0.662 | Temporal: 0.015
9. 230-240 km | Score: 0.349 | Accidents: 173 | Spatial: 0.369 | Severity: 0.673 | Temporal: 0.005
10. 200-210 km | Score: 0.346 | Accidents: 122 | Spatial: 0.260 | Severity: 

In [34]:
weight_configs = [
    (0.5, 0.3, 0.2),
    (0.4, 0.4, 0.2),
    (0.3, 0.5, 0.2),
    (0.6, 0.2, 0.2),
    (0.33, 0.33, 0.34)
]

for spatial_w, severity_w, temporal_w in weight_configs:

    score = (
        spatial_w * concentration_score
        + severity_w * adjusted_severity_score
        + temporal_w * adjusted_temporal_score
    )

    ranking = np.argsort(score)[::-1]

    print(
        f"\nWeights → Spatial={spatial_w}, "
        f"Severity={severity_w}, "
        f"Temporal={temporal_w}"
    )

    for rank, index in enumerate(ranking[:5], start=1):
        start_km = segment_values[index] * 10
        end_km = start_km + 10

        print(
            f"{rank}. {start_km:.0f}-{end_km:.0f} km"
            f" | Score: {score[index]:.3f}"
        )


Weights → Spatial=0.5, Severity=0.3, Temporal=0.2
1. 490-500 km | Score: 0.658
2. 500-510 km | Score: 0.432
3. 220-230 km | Score: 0.422
4. 180-190 km | Score: 0.415
5. 210-220 km | Score: 0.403

Weights → Spatial=0.4, Severity=0.4, Temporal=0.2
1. 490-500 km | Score: 0.610
2. 220-230 km | Score: 0.448
3. 240-250 km | Score: 0.447
4. 500-510 km | Score: 0.443
5. 180-190 km | Score: 0.440

Weights → Spatial=0.3, Severity=0.5, Temporal=0.2
1. 490-500 km | Score: 0.563
2. 240-250 km | Score: 0.497
3. 210-220 km | Score: 0.476
4. 190-200 km | Score: 0.474
5. 220-230 km | Score: 0.474

Weights → Spatial=0.6, Severity=0.2, Temporal=0.2
1. 490-500 km | Score: 0.705
2. 500-510 km | Score: 0.422
3. 220-230 km | Score: 0.397
4. 180-190 km | Score: 0.390
5. 210-220 km | Score: 0.367

Weights → Spatial=0.33, Severity=0.33, Temporal=0.34
1. 490-500 km | Score: 0.504
2. 220-230 km | Score: 0.370
3. 240-250 km | Score: 0.369
4. 500-510 km | Score: 0.367
5. 180-190 km | Score: 0.365
